# SARIMA Inference

This notebook loads the registered SARIMA pipeline from W&B Model Registry and predicts directly on raw `test.csv`. The preprocessing/allocation logic is inside the pipeline object.

In [ ]:
%pip install -q "numpy>=1.24,<3" "pandas>=2.0,<3" "statsmodels>=0.14,<1" "wandb>=0.19,<1" "cloudpickle>=3.0,<4"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import json

import cloudpickle
import numpy as np
import pandas as pd
import wandb

WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
REGISTRY_ARTIFACT_URI = "wandb-registry-model/Walmart_SARIMA_Pipeline:champion"
RUN_NAME = "SARIMA_Registry_Inference_Submission"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/walmart_competition_data"),
    Path("/content/Walmart-Recruiting---Store-Sales-Forecasting/data"),
    Path("../../../../data"),
    Path("../../../data"),
    Path("data"),
]


def resolve_data_dir(candidates):
    required = ["test.csv"]
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError("Could not find test.csv. Update DATA_DIR_CANDIDATES.")


def find_pipeline_file(artifact_dir):
    candidates = sorted(Path(artifact_dir).rglob("*.pkl"))
    if not candidates:
        raise FileNotFoundError(f"No .pkl pipeline file found in {artifact_dir}")
    preferred = [path for path in candidates if "pipeline" in path.name.lower()]
    return preferred[0] if preferred else candidates[0]


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f"Using data directory: {DATA_DIR.resolve()}")

In [ ]:
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
test_raw = test_raw.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)

print({"test_rows": len(test_raw), "test_columns": list(test_raw.columns)})
display(test_raw.head())

## Download Registered SARIMA Pipeline

In [ ]:
run = wandb.init(project=WANDB_PROJECT, name=RUN_NAME, job_type="inference", reinit=True)
model_artifact = run.use_artifact(REGISTRY_ARTIFACT_URI, type="model")
artifact_dir = model_artifact.download()
pipeline_path = find_pipeline_file(artifact_dir)

with pipeline_path.open("rb") as file:
    pipeline = cloudpickle.load(file)

print(f"Loaded pipeline: {pipeline_path}")
print(getattr(pipeline, "metadata", {}))

In [ ]:
predictions = np.asarray(pipeline.predict(test_raw), dtype=float)
predictions = np.clip(predictions, 0, None)

if len(predictions) != len(test_raw):
    raise ValueError(f"Prediction row mismatch: {len(predictions)} != {len(test_raw)}")
if not np.isfinite(predictions).all():
    raise ValueError("Predictions contain non-finite values.")

prediction_summary = {
    "prediction_min": float(predictions.min()),
    "prediction_mean": float(predictions.mean()),
    "prediction_max": float(predictions.max()),
    "prediction_std": float(predictions.std()),
    "submission_rows": int(len(predictions)),
}
prediction_summary

In [ ]:
submission = pd.DataFrame({
    "Id": test_raw["Store"].astype(str) + "_" + test_raw["Dept"].astype(str) + "_" + test_raw["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": predictions,
})

submission_path = OUTPUT_DIR / "sarima_registry_submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Saved {submission_path} with {len(submission)} rows")
display(submission.head())

In [ ]:
run.log({f"inference/{key}": value for key, value in prediction_summary.items()})
run.summary.update(prediction_summary)
run.summary["registry_artifact"] = REGISTRY_ARTIFACT_URI
run.summary["pipeline_file"] = str(pipeline_path)

submission_artifact = wandb.Artifact(
    name="sarima-registry-submission",
    type="submission",
    metadata={**prediction_summary, "registry_artifact": REGISTRY_ARTIFACT_URI},
)
submission_artifact.add_file(str(submission_path))
run.log_artifact(submission_artifact)
run.finish()

print("SARIMA inference complete: registry pipeline -> raw test -> submission artifact.")